# FastAPI Basics

**Project:** Drug Interaction API  
**Goal:** Learn FastAPI by building the backend that will later power the real drug-interaction application.


What is an API? 
- A contract through which one program can communicate with another. 

For our example project. 

```text
UI / Client
    ↓ HTTP request
FastAPI
    ↓
Python business logic
    ↓
Data / Database / ML model
    ↓
JSON response
    ↓
UI / Client
```

A typical request contains:
- HTTP  method :`GET`, `POST`, `PUT`, `PATCH`, `DELETE`
- URL/path 
- optional query parameters
- optional request body
- header

# 2. HTTP methods - Knowing the intent``

| Method | Typical use |
|---|---|
| GET | Read data |
| POST | Create / submit data / trigger an operation |
| PUT | Replace an existing resource |
| PATCH | Partially update a resource |
| DELETE | Delete a resource |


For our API, we'll frequently use requests like 

```text
GET  /drugs
GET  /interactions/Aspirin
POST /check-interaction
```

#### 3. Install FastAPI

```bash
pip install fastapi uvicorn
```

# 4. Let's build something. 

In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Drug Interaction API is running"}


### Run it

From the project root:

```bash
uvicorn app.main:app --reload
```

Meaning:

```text
app.main  → app/main.py
app       → FastAPI object inside main.py
--reload  → restart during development when code changes
```

Open:

```text
http://127.0.0.1:8000/
```


# 5. Automatic API Documentation. 
fastapi automatically provides interactive documentation

Open
```text
http://127.0.0.1:8000/docs
```


# 6. GET endpoint and response data. 

In [ ]:
from fastapi import FastAPI

app = FastAPI()

drugs = [
    "Paracetamol",
    "Aspirin",
    "Warfarin",
    "Atorvastatin",
]


@app.get("/drugs")
def get_drugs():
    return drugs


`http://127.0.0.1:8000/drugs`
shows the 
```text
[
  "Paracetamol",
  "Aspirin",
  "Warfarin",
  "Atorvastatin"
]
```

# 7. Path parameters
A path parameter lets part of the URL become an input to the function. 
Example

```
GET / drugs / Aspril
```

In [ ]:
@app.get("/drugs/{drug_name}")
def get_drug(drug_name: str):
    return {"drug": drug_name}


```http://127.0.0.1:8000/drugs/Aspirin```
returns
```
{"drug":"Aspirin"}
```


The `: str` is a type hint. FastAPI uses type information for validation and API schema generation.


# 8. Query Parameter.
Path parameter is   
```/drugs/Aspirin```

Query parameter is  
```/drugs?name=Aspirin```

In [ ]:
@app.get("/search")
def search_drugs(name: str):
    return {"search": name}


calling at  
```
GET / search?name=Aspirin  

http://127.0.0.1:8000/search?name=Aspirin

```
Gives 
```
{
    "search": "Aspirin"
}
```
Query parameters are useful for filtering, sorting, pagination, and optional behaviour. 

# 9. Request bodies with Pydantic
when the client sends structured JSON, define the expected shape.

In [ ]:
from pydantic import BaseModel

class InteractionRequest(BaseModel):
    drug_a: str
    drug_b: str


A client can send:

```json
{
  "drug_a": "Aspirin",
  "drug_b": "Warfarin"
}
```

FastAPI/Pydantic validates the structure before our endpoint logic runs.

### Mental Model

```text
JSON request
    ↓
Pydantic model
    ↓
Validation
    ↓
Python object
    ↓
Endpoint function
```
- This is safer than manually reading raw Json everywhere. 

# 10. Validation

Pydantic can express constraints. 

In [ ]:
# Validation: pydantic can express constraints. 
from pydantic import BaseModel, Field

class DrugRequest(BaseModel):
    drug_name : str = Field(min_length= 2, max_length= 100)
    

In [ ]:
# Validation: pydantic can express constraints. 
from pydantic import BaseModel, Field

class DrugRequest(BaseModel):
    drug_name : str = Field(min_length= 2, max_length= 100)

@app.post("/check-drugName")
def check_drugName(request: DrugRequest):
    return {"drug_name_received": request.drug_name}


```http://127.0.0.1:8000/check-drugName```
this @ POST gives the output. 


If invalid input is sent, FASTAPI returns a validation error rather than silently passing bad data into the application.

This is an important concept we generally follow in production.. 

# 11. HTTP status Codes. 

Common codes to know:

| Code | Meaning |
|---|---|
| 200 | Successful request |
| 201 | Resource created |
| 204 | Successful, no response body |
| 400 | Bad request |
| 401 | Not authenticated |
| 403 | Authenticated but not allowed |
| 404 | Resource not found |
| 422 | Validation error |
| 500 | Server-side error |


FastAPI commonly returns `422` when request validation fails. 

In [ ]:
from fastapi import status


@app.post("/drugs", status_code=status.HTTP_201_CREATED)
def create_drug(drug: DrugRequest):
    return {"drug": drug.drug_name}



# 12. Returning a controlled response shape

for larger application, define response models too. 

In [ ]:
from pydantic import BaseModel
class DrugResponse(BaseModel):
    name: str 
    available: bool 

@app.get("/drug-example", response_model= DrugResponse)
def drug_example():
    return {
        "name": "Aspirin", 
        "available": True
    }


This is useful because an API should control not only what it accepts, but also what it exposes.

# 13. Small real interaction exmple.

In [ ]:
interactions = [
    {
        "drug_a": "Aspirin",
        "drug_b": "Warfarin",
        "level": "Major"
    },
    {
        "drug_a": "Aspirin",
        "drug_b": "Ibuprofen",
        "level": "Major"
    },
    {
        "drug_a": "Atorvastatin",
        "drug_b": "Clarithromycin",
        "level": "Major"
    }
]


@app.get("/interactions/{drug_name}")
def get_interactions(drug_name: str):
    results = []

    for interaction in interactions:
        if (
            interaction["drug_a"].lower() == drug_name.lower()
            or interaction["drug_b"].lower() == drug_name.lower()
        ):
            results.append(interaction)

    return {
        "drug": drug_name,
        "interactions": results
    }


# 14. Dependency Ingestion 

FastAPI has a dependency system  
A dependency is reusable logic that an endpoint needs.  

In [ ]:
from fastapi import Depends


def get_project_name():
    return "Drug Interaction API"


@app.get("/project")
def project_info(project_name: str = Depends(get_project_name)):
    return {"project": project_name}


In real applications, dependencies are commonly used for:

- database sessions
- authentication
- authorization
- reusable request logic
- configuration
- shared services

# 15. Async


FastAPI supports both:

```python
def endpoint():
    ...
```

and:

```python
async def endpoint():
    ...
```
`async` is useful when the work involves waiting for asynchronous I/O, such as supported database/network operations. 

It mainly helps with efficiently handling `waiting I/O.`

# 16. Middleware

Middleware runs around requests. 


Conceptually:

```text
Request
   ↓
Middleware
   ↓
Route
   ↓
Middleware
   ↓
Response
```

Typical uses:

- CORS
- logging
- request timing
- headers
- tracing

# 17. CORS

If our UI is hosted on a different origin from the API, the browser may enforce CORS rule. 
`cross-origin resource sharing`

Example:  

```py 
UI:
https://my-ui.example

API:
https://my-api.example
```

# Quick Questions::--- 

### 1. what is FastAPI? 
- a python web framework for building APIs, based on Starlette for web capabilities and Pydantic for data validation/schema handling. 

### 2. Why popular ? 

Common reasons include:

- Python type hints
- automatic request validation
- automatic OpenAPI documentation
- good support for async I/O
- relatively small and readable API definitions

### 3. Difference between path and query parameter ? 
```text
/drugs/Aspirin
       ↑
path parameter

/drugs?name=Aspirin
       ↑
query parameter
```

### 4. What does Pydantic do?

It validates and parses structured data based on Python model definitions.

### Q5. Is async always faster?

No. It mainly helps when the application spends significant time waiting for asynchronous I/O. CPU-bound work is a different problem.


### Q6. What happens when request data is invalid?

FastAPI/Pydantic validates the request before the endpoint logic executes and returns a structured validation error.


### 7. Why shouldn't we put everythign in `main.py`? 
Separation of routing, schema, business logic, data acces, configuration, tests,, easier to maintain and test